In [ ]:
import os
import sys
import cv2
import glob
from PIL import Image
import  matplotlib.pyplot as plt

# For data manipulation
import numpy as np
import pandas as pd

# Pytorch Imports
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Utils
from tqdm.auto import tqdm

# Albumentations for augmentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [ ]:
def get_test_transform(img_size):
    return A.Compose([
        # A.Resize(img_size, img_size),
        A.SmallestMaxSize(max_size=img_size, interpolation=3, p=1),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(p=1.0),
    ])

class Customize_Dataset(Dataset):
    def __init__(self, df, transforms):
        self.df = df
        self.image_path = df['image_path'].values
        self.transforms = transforms
    
    def __getitem__(self, index):
        path = self.image_path[index]
        
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        img_224 = self.transforms(img_size=896)(image=img)["image"]
        
        return {
            'image_224': torch.tensor(img_224, dtype=torch.float32),
        }
    
    def __len__(self):
        return len(self.df)

## CFG

In [ ]:
CFG= {
    'img_size': [
        224,
    ],
    'model': [
        '/kaggle/input/lb70-5fold',
    ],
    'model_weight': [
        1.0,
    ],
    
    'TTA': 1,
}

## load model
Models= []
for i in range(len(CFG['model'])):
    models= []
    for m in glob.glob(CFG['model'][i]+'/**'):
        if 'ts' in m:
            models.append( torch.jit.load(m, map_location= 'cuda:0') )
        else:
            models.append( torch.load(m, map_location= 'cuda:0') )
    Models.append(models)
CFG['model']= Models
print(f"length of model: {len(Models)}")

## Create Dataset

In [ ]:
target = [
    'Dry_Clover_g', 
    'Dry_Dead_g', 
    'Dry_Green_g', 
    'Dry_Total_g', 
    'GDM_g'
]

test_df = pd.DataFrame()
imgs = glob.glob('/kaggle/input/csiro-biomass/test/**/*jpg', recursive=True)
test_df['image_path'] = imgs
test_df["name"] = test_df["image_path"].apply(lambda x: x.split("/")[-1])

test_dataset= Customize_Dataset(test_df, get_test_transform)
test_loader= DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)
test_df


## Inference

In [ ]:
def inference(model, img):
    img= torch.unsqueeze(img, 0)
    
    for i, m in enumerate(model):
        with torch.no_grad():
            m.eval()
            if CFG['TTA']:
                imgs= torch.cat([
                            img, 
                            img.flip(-1), 
                            img.flip(-2), 
                            img.flip(-1).flip(-2)
                        ], dim=0)
                
                ## tensor_trt can't use bs!=1
                for j in range(CFG['TTA']):
                    p= m(imgs[j:j+1])
                    if j==0: ps= p
                    else: ps+= p
                pred= ps/CFG['TTA']
            else:
                pred= m(img)[0]
        
        if i==0: preds= pred
        else: preds+= pred
        
    pred= preds/len(model)
    pred= pred.cpu().numpy()
    pred= pred[0]
    return pred

In [ ]:
count= 0
for i, data in enumerate(tqdm(test_loader)):
    
    ## loop for a batch
    for j in range(len(data['image_224'])):
        
        ## multi-model predict
        for indx, m in enumerate(CFG['model']):
            img_size= CFG['img_size'][indx]
            img= data[f'image_{img_size}'][j]
            
            pred= inference(m, img.cuda())
            pred*= CFG['model_weight'][indx]
            
            if indx==0: preds= pred
            else: preds+= pred
        
        test_df.loc[count, 'pred_prob']= str(preds.tolist())
        count+= 1
test_df.head()

## Submission

In [ ]:
submission = pd.DataFrame(columns=('sample_id','target'))
count = 0
for i in range(len(test_df)):
    name = test_df.loc[i,'name'].split('.jpg')[0]
    pred = eval(test_df.loc[i,'pred_prob'])
    for j,t in enumerate(target):
        pred[j] = 0 if pred[j] < 0 else pred[j]
        submission.loc[count, 'sample_id'] = f'{name}__{t}'
        submission.loc[count, 'target'] = pred[j]
        count += 1

    # submission.loc[count, 'sample_id'] = f'{name}__Dry_Clover_g' 
    # submission.loc[count, 'target'] = submission.loc[submission['sample_id']==f'{name}__GDM_g', 'target'].values[0] - submission.loc[submission['sample_id']==f'{name}__Dry_Green_g', 'target'].values[0]
    # count += 1
    # submission.loc[count, 'sample_id'] = f'{name}__Dry_Dead_g' 
    # submission.loc[count, 'target'] = submission.loc[submission['sample_id']==f'{name}__Dry_Total_g', 'target'].values[0] - submission.loc[submission['sample_id']==f'{name}__GDM_g', 'target'].values[0]
    # count += 1


submission.to_csv('submission_richard.csv', index=False)
submission